# xainarratives Quickstart: End-to-End on German Credit

This notebook walks the full `xainarratives` pipeline on a real ML model:

1. **Load** a pre-sampled 30-row slice of the OpenML German Credit dataset.
2. **One-hot encode** categorical columns (20 raw -> 57 features).
3. **Train** a `RandomForestClassifier` and compute SHAP attributions via `TreeExplainer`.
4. **Build** a `TabularExplanationRequest` from the SHAP values via `from_feature_importance`.
5. **Generate** a natural-language narrative with `AnthropicProvider`, **extracting** structured claims in the same call.
6. **Score** the extraction (sign / value / rank fidelity, coverage, hallucination count, readability, perplexity).
7. **Score** narrativity (the seven Cedro & Martens 2026 paper metrics) using a Together-hosted perplexity provider.

**Outputs are illustrative.** LLM responses are non-deterministic, and the perplexity / narrativity numbers shift run-to-run depending on the provider's tokenization and the model's logprob output. Treat the numbers in the rendered output as one realization, not as canonical ground truth.

## Prerequisites

````bash
pip install -e ".[notebook,anthropic,perplexity-api]"
export ANTHROPIC_API_KEY="sk-ant-..."
export TOGETHER_API_KEY="..."
```

In [1]:
import os

import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier

from xainarratives import (
    DatasetSchema,
    Explainer,
    ExplanationConfig,
    FeatureSchema,
    Modality,
    Prediction,
    TargetSchema,
    score_extraction,
    score_narrativity,
)
from xainarratives.integrations import from_feature_importance
from xainarratives.metrics import OpenAICompatibleEchoProvider
from xainarratives.prompts import FactualTabularPromptTemplate
from xainarratives.config import DEFAULT_NARRATIVE_RULES
from xainarratives.providers import AnthropicProvider

if "ANTHROPIC_API_KEY" not in os.environ:
    raise KeyError("Set ANTHROPIC_API_KEY (Anthropic Claude API key).")
if "TOGETHER_API_KEY" not in os.environ:
    raise KeyError("Set TOGETHER_API_KEY (Together.ai API key for perplexity).")

## Step 1 - Load the German Credit slice

We use a pre-sampled 30-row slice committed at `notebooks/data/german_credit_sample.csv`. The original dataset is OpenML's `credit-g` (1000 rows, 20 features); the slice was generated with random seed 42 - see `scripts/generate_german_credit_sample.py` for regeneration. The CSV preserves the raw schema: categorical columns are strings, numeric columns are integers, and the `class` target is mapped from `"good"`/`"bad"` to `0`/`1`.

In [2]:
df = pd.read_csv("data/german_credit_sample.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (30, 21)
Columns: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker', 'class']


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,18,existing paid,radio/tv,3190,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,24,none,own,1,skilled,1,none,yes,1
1,<0,18,existing paid,new car,4380,100<=X<500,1<=X<4,3,male single,none,...,car,35,none,own,1,unskilled resident,2,yes,yes,0
2,<0,24,all paid,new car,2325,100<=X<500,4<=X<7,2,male single,none,...,car,32,bank,own,1,skilled,1,none,yes,0
3,>=200,12,existing paid,radio/tv,1297,<100,1<=X<4,3,male mar/wid,none,...,real estate,23,none,rent,1,skilled,1,none,yes,0
4,no checking,33,critical/other existing credit,used car,7253,<100,4<=X<7,3,male single,none,...,car,35,none,own,2,high qualif/self emp/mgmt,1,yes,yes,0


## Step 2 - One-hot encode

`pd.get_dummies(..., drop_first=False)` expands the 13 categorical columns into binary indicator columns, producing 57 features in total. `xainarratives` doesn't impose a feature representation - whatever your upstream encoder produces is what flows through the pipeline.

In [3]:
y = df["class"]
X_raw = df.drop(columns=["class"])
X = pd.get_dummies(X_raw, drop_first=False)
print(f"Raw features: {X_raw.shape[1]} -> one-hot features: {X.shape[1]}")
X.head()

Raw features: 20 -> one-hot features: 57


,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,checking_status_0<=X<200,checking_status_<0,checking_status_>=200,...,housing_own,housing_rent,job_high qualif/self emp/mgmt,job_skilled,job_unemp/unskilled non res,job_unskilled resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes
0,18,3190,2,2,24,1,1,False,True,False,...,True,False,False,True,False,False,True,False,False,True
1,18,4380,3,4,35,1,2,False,True,False,...,True,False,False,False,False,True,False,True,False,True
2,24,2325,2,3,32,1,1,False,True,False,...,True,False,False,True,False,False,True,False,False,True
3,12,1297,3,4,23,1,1,False,False,True,...,False,True,False,True,False,False,True,False,False,True
4,33,7253,3,2,35,2,1,False,False,False,...,True,False,True,False,False,False,False,True,False,True


## Step 3 - Train a RandomForestClassifier and compute SHAP

We fit `RandomForestClassifier(random_state=42)` on the full 30-row slice and compute SHAP values via `shap.TreeExplainer` (fast, exact for tree models). For binary classification, SHAP >= 0.46 returns a 3D array of shape `(n_samples, n_features, n_classes)`; we slice on the positive-class axis (index 1) to get per-instance per-feature contributions to the `class=1` ("bad credit risk") prediction.

This notebook is about **verbalization**, not model evaluation - training and explaining on the same slice is fine here, and we'd never do it for real.

In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

tree_explainer = shap.TreeExplainer(model)
shap_values_3d = tree_explainer.shap_values(X)   # shape (30, 57, 2)
shap_values = shap_values_3d[:, :, 1]             # positive-class slice; shape (30, 57)

print(f"SHAP matrix shape: {shap_values.shape}")
print(f"Model accuracy on the slice: {model.score(X, y):.2f}")

SHAP matrix shape: (30, 57)
Model accuracy on the slice: 1.00


## Step 4 - Pick one instance and build the explanation request

We pick **row 2** to explain end-to-end. The model predicts `0` (Good credit risk) with `P(bad) = 0.16`. The top-5 SHAP contributions (by absolute magnitude) are:

| feature | SHAP | direction |
|---|---|---|
| `checking_status_<0` | +0.1032 | toward "bad" |
| `savings_status_<100` | -0.0454 | toward "good" |
| `checking_status_no checking` | +0.0264 | toward "bad" |
| `credit_amount` | -0.0240 | toward "good" |
| `age` | -0.0222 | toward "good" |

Two positive, three negative, magnitude spread of ~0.08. The dominant signal (a negative checking balance pushing toward "bad" risk) is intuitively plausible - a useful property for a tutorial reader's trust - and the mixed signs give the narrative real push-pull.

The `from_feature_importance` adapter receives **all 57** feature values + **all 57** SHAP scalars and emits all 57 contributions on the request. The top-k selection happens later, inside the prompt template - `ExplanationConfig.top_k_features` controls how many contributions the LLM actually sees.

In [5]:
instance_idx = 2

# Pull everything we need for this instance.
instance_features = {col: float(X.iloc[instance_idx][col]) for col in X.columns}
proba = model.predict_proba(X.iloc[[instance_idx]])[0]
predicted_class = int(model.classes_[int(proba.argmax())])
probabilities = {int(c): float(p) for c, p in zip(model.classes_, proba)}
importances = {col: float(shap_values[instance_idx, j]) for j, col in enumerate(X.columns)}

print(f"Instance row {instance_idx}: predicted_class={predicted_class}, probabilities={probabilities}")

# Build the schema. get_dummies leaves integer columns untouched and only expands
# categoricals, so X.columns intersected with X_raw.columns is exactly the original
# numeric features; everything else is a 0/1 one-hot indicator.
features_schema = [
    FeatureSchema(
        name=col,
        dtype="numeric" if col in X_raw.columns else "boolean",
        description=(
            f"Original numeric feature: {col}"
            if col in X_raw.columns
            else f"One-hot indicator: {col}"
        ),
    )
    for col in X.columns
]

schema = DatasetSchema(
    modality=Modality.TABULAR,
    name="german_credit",
    description="Predicts whether a loan applicant is a good or bad credit risk (OpenML credit-g).",
    target=TargetSchema(
        name="class",
        description="Credit risk classification.",
        classes={0: "Good credit risk", 1: "Bad credit risk"},
    ),
    features=features_schema,
)

request = from_feature_importance(
    features=instance_features,
    importances=importances,
    prediction=Prediction(predicted_class=predicted_class, probabilities=probabilities),
    instance_id=f"row_{instance_idx}",
)
print(f"Request carries {len(request.contributions)} contributions (all encoded features).")

Instance row 2: predicted_class=0, probabilities={0: 0.84, 1: 0.16}
Request carries 57 contributions (all encoded features).


## Step 5 - Generate the narrative and extract structured claims

`Explainer.explain()` makes **two** LLM calls under the hood:

1. **Generator** - `AnthropicProvider` produces the narrative. The prompt template (`FactualTabularPromptTemplate`) selects the **top-5** contributions by `|importance|` per `ExplanationConfig.top_k_features=5`; only those reach the LLM.
2. **Judge / extractor** - the same provider (or a separately-configured `judge_llm`) is called again to parse the narrative into structured `FeatureClaim`s, resolving each mention to a schema feature name or marking it as a hallucination.

Both happen automatically; we receive a single `ExplanationResult` carrying the narrative, the extraction, and token / latency / guardrail audit metadata.

The prompt the generator receives is shaped entirely by `ExplanationConfig`. Beyond `audience`, `tone`, and `top_k_features`, the `narrative_rules` field carries the operational definition of an XAI Narrative that is injected into every prompt. It is a plain string -- pass your own to `ExplanationConfig` to change how the narrative is written, and the same rules apply to all narrative modes (factual, contrastive, counterfactual).

In [6]:
llm = AnthropicProvider(model="claude-haiku-4-5", max_tokens=512)

explainer = Explainer(
    schema=schema,
    llm=llm,
    # judge_llm defaults to llm; shown explicitly to mark it as a knob.
    judge_llm=llm,
    prompt_template=FactualTabularPromptTemplate(),
    # Every field below is a knob -- change them to reshape the narrative.
    config=ExplanationConfig(
        mode="factual",
        audience="end_user",
        language="en",
        tone="neutral",
        top_k_features=5,
        extract_narrative=True,
        narrative_rules=DEFAULT_NARRATIVE_RULES,
    ),
)
result = explainer.explain(request)

print("=== Narrative ===")
print(result.text)
print()
print(f"Generator tokens: {result.tokens_used}")
print(f"Extraction tokens: {result.guardrail_tokens_used}")
print(f"Latency: {result.latency_ms:.0f} ms")

# Display the extraction joined against ground-truth SHAP for the resolved features.
extraction = result.narrative_extraction
claims_rows = [
    {
        "schema_name": name,
        "narrative_name": claim.narrative_name,
        "narrative_sign": claim.sign,
        "narrative_value": claim.value,
        "shap_importance": importances.get(name),
    }
    for name, claim in extraction.features.items()
]
print(
    f"\nResolved claims: {len(extraction.features)}; "
    f"hallucinations: {len(extraction.hallucinations)}"
)
pd.DataFrame(claims_rows).round(2)

=== Narrative ===
# Why This Applicant Was Classified as Good Credit Risk

Your application received a favorable assessment primarily because your checking account status strongly signals reliability. Having an active checking account with a balance below 100 demonstrates financial engagement, substantially boosting your creditworthiness score by contributing +0.103 points—the most decisive factor in this evaluation.

Conversely, the absence of substantial savings slightly dampened your profile, reducing it by 0.045 points, suggesting limited financial reserves. However, this concern was outweighed by your checking account activity. The requested credit amount of 2,325 and your age of 32 provided minor headwinds, each marginally lowering the assessment, yet remained insufficient to override the positive signal from your active account management.

Ultimately, your demonstrated banking activity pattern proved decisive, positioning you as an acceptable lending candidate despite modest sa

,schema_name,narrative_name,narrative_sign,narrative_value,shap_importance
0,checking_status_<0,checking account status with a balance below 100,1,NaN,0.10
1,savings_status_<100,absence of substantial savings,-1,NaN,-0.05
2,credit_amount,"requested credit amount of 2,325",-1,2325.0,-0.02
3,age,age of 32,-1,32.0,-0.02


## Step 6 - Score the extraction

`score_extraction` returns an `ExtractionScores` model:

- **Fidelity:** as in Ichmoukhamedov et al. (2025): `sign_faithfulness`, `value_faithfulness`, `rank_correlation` - in `[0, 1]` or `[-1, 1]`, `None` when undefined (e.g. fewer than two comparable features for rank correlation).
- **Coverage:** fraction of top-`k` schema features the narrative resolved.
- **Hallucination count:** number of narrative mentions the extractor couldn't resolve.
- **Readability:** Flesch reading-ease of the narrative text.
- **Perplexity:** mean perplexity of the narrative, computed only when a `PerplexityProvider` is supplied.

We construct an `OpenAICompatibleEchoProvider` (Together.ai) here and pass it to `score_extraction` so the `perplexity` field is populated; the same provider is reused for `score_narrativity` in Step 7. We pass `k=5` to align with `top_k_features=5` in the prompt template.

In [7]:
ppl_provider = OpenAICompatibleEchoProvider(
    base_url="https://api.together.xyz/v1",
    api_key=os.environ["TOGETHER_API_KEY"],
    model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
)

ext_scores = score_extraction(
    extraction=extraction,
    request=request,
    schema=schema,
    narrative_text=result.text,
    k=5,
    perplexity_provider=ppl_provider,
)
pd.Series(ext_scores.model_dump())

sign_faithfulness            1.0
value_faithfulness           1.0
rank_correlation        0.989949
coverage                     0.8
hallucination_count            0
readability              13.8375
perplexity             28.841197
prompt_version                 2
dtype: object

## Step 7 - Score narrativity

`score_narrativity` computes the seven paper metrics - CSR, DCPR, CCPR, CECPR, FDR, TTCPR, VCPR - plus nine auxiliary primitives (`ppl_ordered`, `ppl_shuffled`, `decay_constant`, `dist2`, `ttr`, `vr`, `cr`, `cer`, `n_sentences`). It reuses the `OpenAICompatibleEchoProvider` constructed in Step 6.

The provider makes **N + 1** perplexity calls for an N-sentence narrative: one per cumulative prefix (`s1`, `s1+s2`, ..., `s1+...+sN`) plus one for the shuffled-sentences variant (CSR). `cecpr` is `None`/`NaN` when the narrative contains no explicit cause-effect markers - expected for terse explanations.

In [8]:
narr_scores = score_narrativity(result.text, ppl_provider)
pd.Series(narr_scores.model_dump())

csr                   0.189322
dcpr                  1.157680
ccpr                180.819159
cecpr             18081.915942
fdr                   0.280603
ttcpr                 2.046392
vcpr                 41.002077
ppl_ordered          29.885519
ppl_shuffled         35.543491
decay_constant        1.103633
dist2                 0.976378
ttr                   0.734375
vr                    0.164062
cr                    0.078125
cer                   0.007812
n_sentences           6.000000
dtype: float64

## Summary

What we built end-to-end:

1. Loaded a 30-row German Credit slice.
2. One-hot encoded 20 raw features -> 57 features.
3. Trained a RandomForest and computed SHAP attributions.
4. Built a `TabularExplanationRequest` from all 57 SHAP scalars (top-k selection deferred to the prompt).
5. Generated a natural-language narrative + extracted structured claims (two LLM calls via `Explainer.explain`).
6. Scored the extraction along fidelity, coverage, hallucination, readability, perplexity axes.
7. Scored narrativity (7 paper metrics + auxiliary primitives) using a perplexity provider.

**Cost per full run:** ~$0.001-0.005 (Haiku generation + extraction + Together perplexity calls).